# Prapemrosesan Data & Augmentasi Tingkat Lanjut

In [1]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedKFold
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==========================================
# 1. KONFIGURASI JALUR & HIPERPARAMETER
# ==========================================
BASE_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026"
TRAIN_DIR = os.path.join(BASE_PATH, "train")

IMG_SIZE = 384 
BATCH_SIZE = 8
N_FOLDS = 5 

# ==========================================
# 2. PEMBUATAN DATAFRAME & STRATIFIED K-FOLD
# ==========================================
class_names = sorted(os.listdir(TRAIN_DIR))
label2id = {class_name: i for i, class_name in enumerate(class_names)}
id2label = {i: class_name for class_name, i in label2id.items()}

data = []
for class_name in class_names:
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            data.append({
                "image_path": os.path.join(class_dir, img_name),
                "label": label2id[class_name]
            })

df_all = pd.DataFrame(data)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
df_all['fold'] = -1

for fold, (train_idx, valid_idx) in enumerate(skf.split(X=df_all, y=df_all['label'])):
    df_all.loc[valid_idx, 'fold'] = fold

# ==========================================
# 3. PIPELINE AUGMENTASI (ALBUMENTATIONS)
# ==========================================
train_transforms = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    
    # --- REVISI: AUGMENTASI EKSTREM KHUSUS SPOOFING ---
    # Terapkan secara acak 1 dari 4 gangguan di bawah ini pada 50% gambar latih
    A.OneOf([
        A.ImageCompression(quality_lower=60, quality_upper=95, p=1.0), 
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),                      
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0), 
        A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),                   
    ], p=0.5),
    # --------------------------------------------------
    
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.5),
    
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, p=0.5),
    
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

valid_transforms = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 4. KELAS PYTORCH DATASET
# ==========================================
class SpoofingDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image = cv2.imread(img_path)
        
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']

        label = self.df.loc[idx, 'label']
        return image, torch.tensor(label, dtype=torch.long)

print("Prapemrosesan dan Pembagian K-Fold (dengan Augmentasi Ekstrem) Selesai!")
print(f"Total Seluruh Data : {len(df_all)} gambar")

Prapemrosesan dan Pembagian K-Fold (dengan Augmentasi Ekstrem) Selesai!
Total Seluruh Data : 1652 gambar


/tmp/ipykernel_55/1118629522.py:56: UserWarning: Argument(s) 'quality_lower, quality_upper' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=60, quality_upper=95, p=1.0),
/tmp/ipykernel_55/1118629522.py:59: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),


# Membangun Arsitektur Swin dan CNN

In [2]:
import torch
import torch.nn as nn
import timm
import gc

# ==========================================
# 1. KONFIGURASI PERANGKAT (GPU / CPU)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat: {device}")

# ==========================================
# 2. FUNGSI PEMBUAT MODEL (FACTORY FUNCTIONS)
# ==========================================
NUM_CLASSES = 6
MODEL_SWIN_NAME = 'swin_base_patch4_window12_384.ms_in22k'
MODEL_CNN_NAME = 'tf_efficientnetv2_m.in21k_ft_in1k'

# --- PABRIK 1: SWIN TRANSFORMER + ReLU CUSTOM HEAD ---
class SwinWithCustomHead(nn.Module):
    def __init__(self, model_name, num_classes):
        super(SwinWithCustomHead, self).__init__()
        
        # Muat tulang punggung tanpa lapisan akhir bawaan
        self.backbone = timm.create_model(
            model_name, 
            pretrained=True, 
            num_classes=0,       
            drop_rate=0.3,       
            attn_drop_rate=0.2   
        )
        
        num_features = self.backbone.num_features
        
        # Kepala klasifikasi kustom dengan ReLU untuk penalaran non-linear
        self.custom_head = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        output = self.custom_head(features)
        return output

def create_swin_model():
    model = SwinWithCustomHead(MODEL_SWIN_NAME, NUM_CLASSES)
    return model.to(device)


# --- PABRIK 2: EFFICIENTNET V2 ---
def create_cnn_model():
    model = timm.create_model(
        MODEL_CNN_NAME, 
        pretrained=True, 
        num_classes=NUM_CLASSES,
        drop_rate=0.3,       
        drop_path_rate=0.2   
    )
    return model.to(device)


# ==========================================
# 3. UJI COBA FUNGSI (SANITY CHECK)
# ==========================================
print("\nMenguji coba pembuatan kedua model...")
temp_swin = create_swin_model()
temp_cnn = create_cnn_model()

dummy_input = torch.randn(4, 3, 384, 384).to(device)

with torch.no_grad():
    dummy_out_swin = temp_swin(dummy_input)
    dummy_out_cnn = temp_cnn(dummy_input)

print("Fungsi pembuat model berhasil diverifikasi dan siap mencetak 10 model!")
print(f"Bentuk input        : {dummy_input.shape}")
print(f"Bentuk output Swin  : {dummy_out_swin.shape}")
print(f"Bentuk output CNN   : {dummy_out_cnn.shape}")

# Bersihkan memori GPU secara total agar siap untuk pelatihan berat
del temp_swin, temp_cnn
torch.cuda.empty_cache()
gc.collect()

Menggunakan perangkat: cuda

Menguji coba pembuatan kedua model...


model.safetensors:   0%|          | 0.00/451M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/218M [00:00<?, ?B/s]

Fungsi pembuat model berhasil diverifikasi dan siap mencetak 10 model!
Bentuk input        : torch.Size([4, 3, 384, 384])
Bentuk output Swin  : torch.Size([4, 6])
Bentuk output CNN   : torch.Size([4, 6])


249

# Strategi Pelatihan (Warm-up & Loss)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from tqdm import tqdm
import numpy as np
import gc

# ==========================================
# 1. DEFINISI FOCAL LOSS DENGAN LABEL SMOOTHING
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean', label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma 
        self.reduction = reduction
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        num_classes = inputs.size(1)
        smooth_target = F.one_hot(targets, num_classes=num_classes).float()
        smooth_target = smooth_target * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)
        
        log_pt = F.log_softmax(inputs, dim=1)
        pt = torch.exp(log_pt)
        
        ce_loss = -(smooth_target * log_pt).sum(dim=1)
        F_loss = self.alpha * (1 - pt.gather(1, targets.view(-1, 1)).squeeze(1))**self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        return F_loss

# ==========================================
# 2. FUNGSI PELATIHAN GENERIC
# ==========================================
def train_fold(fold, train_loader, valid_loader, model_func, model_prefix, epochs=30, lr=1e-5, patience=10, accumulation_steps=4):
    print(f"\n{'='*50}")
    print(f"Memulai Pelatihan {model_prefix}: FOLD {fold}")
    print(f"{'='*50}")
    
    model = model_func()
    
    criterion = FocalLoss(gamma=2.0, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler()

    best_val_f1 = 0.0
    patience_counter = 0
    save_path = f"best_{model_prefix}_fold_{fold}.pth"

    for epoch in range(epochs):
        # --- FASE TRAINING ---
        model.train()
        train_loss = 0.0
        optimizer.zero_grad() 
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train {model_prefix} Fold {fold}]")
        for batch_idx, (images, labels) in enumerate(train_bar):
            images, labels = images.to(device), labels.to(device)
            
            with torch.cuda.amp.autocast():
                outputs = model(images) 
                loss = criterion(outputs, labels) 
                loss = loss / accumulation_steps 
            
            scaler.scale(loss).backward() 
            
            if ((batch_idx + 1) % accumulation_steps == 0) or (batch_idx + 1 == len(train_loader)):
                scaler.step(optimizer) 
                scaler.update()
                optimizer.zero_grad() 
            
            train_loss += loss.item() * accumulation_steps * images.size(0)
            train_bar.set_postfix(loss=(loss.item() * accumulation_steps))
            
        avg_train_loss = train_loss / len(train_loader.dataset)
        
        # --- FASE VALIDASI ---
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []
        
        val_bar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid {model_prefix} Fold {fold}]")
        with torch.no_grad(): 
            for images, labels in val_bar:
                images, labels = images.to(device), labels.to(device)
                
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item() * images.size(0)
                preds = torch.argmax(outputs, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(valid_loader.dataset)
        val_macro_f1 = f1_score(all_labels, all_preds, average='macro')
        
        print(f"\nHasil Epoch {epoch+1} ({model_prefix} Fold {fold}):")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}")
        
        scheduler.step()
        
        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            torch.save(model.state_dict(), save_path)
            print(f"Model membaik! Disimpan ke '{save_path}'")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"Model tidak membaik. Kesabaran: {patience_counter}/{patience}")
            
        if patience_counter >= patience:
            print(f"\nEarly Stopping dipicu untuk {model_prefix} Fold {fold}!")
            break

    print(f"\nPelatihan {model_prefix} Fold {fold} Selesai! Skor Macro F1 Terbaik: {best_val_f1:.4f}")
    
    del model
    torch.cuda.empty_cache()
    gc.collect()
    
    return best_val_f1

# ==========================================
# 3. EKSEKUSI PELATIHAN 10 MODEL (5 SWIN + 5 CNN)
# ==========================================
swin_fold_scores = []
cnn_fold_scores = []

for fold in range(N_FOLDS):
    train_df_fold = df_all[df_all['fold'] != fold].reset_index(drop=True)
    valid_df_fold = df_all[df_all['fold'] == fold].reset_index(drop=True)
    
    train_dataset_fold = SpoofingDataset(train_df_fold, transforms=train_transforms)
    valid_dataset_fold = SpoofingDataset(valid_df_fold, transforms=valid_transforms)
    
    # REVISI: Menambahkan drop_last=True agar sisa batch berukuran 1 dibuang dan tidak merusak BatchNorm
    train_loader_fold = DataLoader(train_dataset_fold, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
    # Validasi tidak perlu drop_last karena mode evaluasi BatchNorm tidak mengupdate statistik
    valid_loader_fold = DataLoader(valid_dataset_fold, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # --- MELATIH SWIN TRANSFORMER ---
    best_swin_f1 = train_fold(
        fold=fold, 
        train_loader=train_loader_fold, 
        valid_loader=valid_loader_fold, 
        model_func=create_swin_model, 
        model_prefix="swin",
        lr=1e-5
    )
    swin_fold_scores.append(best_swin_f1)
    
    # --- MELATIH EFFICIENTNET V2 ---
    best_cnn_f1 = train_fold(
        fold=fold, 
        train_loader=train_loader_fold, 
        valid_loader=valid_loader_fold, 
        model_func=create_cnn_model, 
        model_prefix="cnn",
        lr=2e-5 
    )
    cnn_fold_scores.append(best_cnn_f1)

# ==========================================
# 4. REKAPITULASI AKHIR
# ==========================================
print(f"\n{'='*50}")
print("REKAPITULASI PELATIHAN 10-MODEL ENSEMBLE")
print(f"{'='*50}")
print("--- SWIN TRANSFORMER (CUSTOM HEAD ReLU) ---")
for i, score in enumerate(swin_fold_scores):
    print(f"Fold {i} F1-Score : {score:.4f}")
print(f"Rata-rata Swin CV F1-Score : {np.mean(swin_fold_scores):.4f}\n")

print("--- EFFICIENTNET V2 (CNN) ---")
for i, score in enumerate(cnn_fold_scores):
    print(f"Fold {i} F1-Score : {score:.4f}")
print(f"Rata-rata CNN CV F1-Score  : {np.mean(cnn_fold_scores):.4f}")


Memulai Pelatihan swin: FOLD 0


/tmp/ipykernel_55/3232661585.py:50: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Epoch 1/30 [Train swin Fold 0]:   0%|          | 0/165 [00:00<?, ?it/s]/tmp/ipykernel_55/3232661585.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Valid swin Fold 0]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/3232661585.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Valid swin Fold 0]: 100%|██████████| 42/42 [00:08<00:00,  4.91it/s]



Hasil Epoch 1 (swin Fold 0):
Train Loss: 1.2808 | Val Loss: 0.9713 | Val Macro F1: 0.4597
Model membaik! Disimpan ke 'best_swin_fold_0.pth'


Epoch 2/30 [Train swin Fold 0]:   0%|          | 0/165 [00:00<?, ?it/s]/tmp/ipykernel_55/3232661585.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/30 [Valid swin Fold 0]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/3232661585.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/30 [Valid swin Fold 0]: 100%|██████████| 42/42 [00:07<00:00,  5.26it/s]



Hasil Epoch 2 (swin Fold 0):
Train Loss: 1.0186 | Val Loss: 0.7083 | Val Macro F1: 0.7056
Model membaik! Disimpan ke 'best_swin_fold_0.pth'


...


Hasil Epoch 28 (cnn Fold 4):
Train Loss: 0.6714 | Val Loss: 0.5462 | Val Macro F1: 0.7579
Model membaik! Disimpan ke 'best_cnn_fold_4.pth'


Epoch 29/30 [Train cnn Fold 4]:   0%|          | 0/165 [00:00<?, ?it/s]/tmp/ipykernel_55/3232661585.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 29/30 [Valid cnn Fold 4]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/3232661585.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 29/30 [Valid cnn Fold 4]: 100%|██████████| 42/42 [00:06<00:00,  6.67it/s]



Hasil Epoch 29 (cnn Fold 4):
Train Loss: 0.7141 | Val Loss: 0.5174 | Val Macro F1: 0.7498
Model tidak membaik. Kesabaran: 1/10


Epoch 30/30 [Train cnn Fold 4]:   0%|          | 0/165 [00:00<?, ?it/s]/tmp/ipykernel_55/3232661585.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 30/30 [Valid cnn Fold 4]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/3232661585.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 30/30 [Valid cnn Fold 4]: 100%|██████████| 42/42 [00:06<00:00,  6.67it/s]



Hasil Epoch 30 (cnn Fold 4):
Train Loss: 0.6934 | Val Loss: 0.5287 | Val Macro F1: 0.7542
Model tidak membaik. Kesabaran: 2/10

Pelatihan cnn Fold 4 Selesai! Skor Macro F1 Terbaik: 0.7579

REKAPITULASI PELATIHAN 10-MODEL ENSEMBLE
--- SWIN TRANSFORMER (CUSTOM HEAD ReLU) ---
Fold 0 F1-Score : 0.8338
Fold 1 F1-Score : 0.8374
Fold 2 F1-Score : 0.8091
Fold 3 F1-Score : 0.8522
Fold 4 F1-Score : 0.8697
Rata-rata Swin CV F1-Score : 0.8404

--- EFFICIENTNET V2 (CNN) ---
Fold 0 F1-Score : 0.7219
Fold 1 F1-Score : 0.6896
Fold 2 F1-Score : 0.7147
Fold 3 F1-Score : 0.7321
Fold 4 F1-Score : 0.7579
Rata-rata CNN CV F1-Score  : 0.7232


# Kalibrasi Probabilitas (Thresholding)

In [5]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize
from torch.utils.data import DataLoader
import warnings
import gc

# Mengabaikan warning scipy agar output terminal tetap bersih
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 6
N_FOLDS = 5 

# Menyiapkan array kosong untuk menampung probabilitas (Swin, CNN, dan Gabungan)
oof_probs_swin = np.zeros((len(df_all), NUM_CLASSES))
oof_probs_cnn = np.zeros((len(df_all), NUM_CLASSES))
oof_labels = np.zeros(len(df_all))

# ==========================================
# 1. EKSTRAKSI PROBABILITAS OUT-OF-FOLD (OOF)
# ==========================================
print("=== EKSTRAKSI PROBABILITAS OUT-OF-FOLD (10 MODEL) ===")

for fold in range(N_FOLDS):
    print(f"\nMengekstrak prediksi dari Fold {fold}...")
    
    # Ambil data validasi khusus untuk fold ini
    val_idx = df_all[df_all['fold'] == fold].index
    valid_df_fold = df_all.iloc[val_idx].reset_index(drop=True)
    
    valid_dataset_fold = SpoofingDataset(valid_df_fold, transforms=valid_transforms)
    valid_loader_fold = DataLoader(valid_dataset_fold, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # --- A. EKSTRAKSI SWIN TRANSFORMER ---
    # Memanggil fungsi pembuat model dari Tahap 2
    model_swin = create_swin_model()
    model_swin.load_state_dict(torch.load(f'best_swin_fold_{fold}.pth', map_location=device))
    model_swin.eval()
    
    fold_probs_swin = []
    fold_labels = []
    
    with torch.no_grad():
        for images, labels in valid_loader_fold: 
            images = images.to(device)
            
            # Swin Prediction
            out_swin = model_swin(images)
            probs_swin = F.softmax(out_swin, dim=1) 
            fold_probs_swin.extend(probs_swin.cpu().numpy())
            fold_labels.extend(labels.numpy())
            
    oof_probs_swin[val_idx] = np.array(fold_probs_swin)
    oof_labels[val_idx] = np.array(fold_labels)
    
    del model_swin
    torch.cuda.empty_cache()
    gc.collect()

    # --- B. EKSTRAKSI EFFICIENTNET V2 ---
    model_cnn = create_cnn_model()
    model_cnn.load_state_dict(torch.load(f'best_cnn_fold_{fold}.pth', map_location=device))
    model_cnn.eval()
    
    fold_probs_cnn = []
    
    with torch.no_grad():
        for images, _ in valid_loader_fold: 
            images = images.to(device)
            
            # CNN Prediction
            out_cnn = model_cnn(images)
            probs_cnn = F.softmax(out_cnn, dim=1) 
            fold_probs_cnn.extend(probs_cnn.cpu().numpy())
            
    oof_probs_cnn[val_idx] = np.array(fold_probs_cnn)
    
    del model_cnn
    torch.cuda.empty_cache()
    gc.collect()

# ==========================================
# 2. PENGGABUNGAN (ENSEMBLE) & EVALUASI AWAL
# ==========================================
# Rasio kekuatan: Swin 60% (karena ada Custom Head), CNN 40% (Spesialis Tekstur)
oof_probs_ensemble = (oof_probs_swin * 0.60) + (oof_probs_cnn * 0.40)

preds_before = np.argmax(oof_probs_ensemble, axis=1)
f1_before = f1_score(oof_labels, preds_before, average='macro')

print("\n--- SEBELUM KALIBRASI (ENSEMBLE 10-MODEL OOF) ---")
print(f"Macro F1-Score Keseluruhan: {f1_before:.5f}")
print(classification_report(oof_labels, preds_before, target_names=class_names, digits=4))

# ==========================================
# 3. PROSES OPTIMASI THRESHOLD (POWELL)
# ==========================================
def optimize_f1(weights):
    weighted_probs = oof_probs_ensemble * weights
    calibrated_preds = np.argmax(weighted_probs, axis=1)
    return -1.0 * f1_score(oof_labels, calibrated_preds, average='macro')

initial_weights = [1.0] * 6 
bounds = [(0.1, 3.0)] * 6 

print("\nMemulai optimasi kalibrasi probabilitas (Metode: Powell)...")
result = minimize(optimize_f1, initial_weights, method='Powell', bounds=bounds)
best_weights = result.x

# ==========================================
# 4. EVALUASI SETELAH KALIBRASI
# ==========================================
weighted_probs_after = oof_probs_ensemble * best_weights
preds_after = np.argmax(weighted_probs_after, axis=1)
f1_after = f1_score(oof_labels, preds_after, average='macro')

print("\n--- SETELAH KALIBRASI ---")
print(f"Macro F1-Score: {f1_after:.5f}")
print(f"Peningkatan   : {f1_after - f1_before:.5f}")
print(classification_report(oof_labels, preds_after, target_names=class_names, digits=4))

print("\nSimpan array bobot ini untuk dipakai di Test Set (Tahap 5):")
print(f"best_weights = {list(best_weights)}")

=== EKSTRAKSI PROBABILITAS OUT-OF-FOLD (10 MODEL) ===

Mengekstrak prediksi dari Fold 0...

Mengekstrak prediksi dari Fold 1...

Mengekstrak prediksi dari Fold 2...

Mengekstrak prediksi dari Fold 3...

Mengekstrak prediksi dari Fold 4...

--- SEBELUM KALIBRASI (ENSEMBLE 10-MODEL OOF) ---
Macro F1-Score Keseluruhan: 0.82083
                precision    recall  f1-score   support

fake_mannequin     0.8340    0.8973    0.8645       224
     fake_mask     0.8699    0.8417    0.8556       278
  fake_printed     0.6555    0.6610    0.6582       118
   fake_screen     0.9275    0.7817    0.8483       229
  fake_unknown     0.8289    0.8314    0.8301       338
    realperson     0.8452    0.8925    0.8682       465

      accuracy                         0.8402      1652
     macro avg     0.8268    0.8176    0.8208      1652
  weighted avg     0.8424    0.8402    0.8400      1652


Memulai optimasi kalibrasi probabilitas (Metode: Powell)...

--- SETELAH KALIBRASI ---
Macro F1-Score: 0.83641

# Prediksi TTA & Pengumpulan (Submission)

In [6]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import warnings
import gc

warnings.filterwarnings("ignore")

# ==========================================
# 1. PERSIAPAN JALUR & TEMPLATE SUBMISSION
# ==========================================
TEST_DIR = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/samplesubmission.csv"

sub_df = pd.read_csv(SAMPLE_SUB_PATH)

class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
id2label = {i: class_name for i, class_name in enumerate(class_names)}

IMG_SIZE = 384
BATCH_SIZE = 8 
N_FOLDS = 5

test_transforms = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 2. DATASET ANTI-CRASH (DETEKSI OTOMATIS)
# ==========================================
class TestDatasetTTA(Dataset):
    def __init__(self, df, test_dir, transforms):
        self.df = df
        self.test_dir = test_dir
        self.transforms = transforms
        
        self.existing_files = {}
        if os.path.exists(test_dir):
            for f in os.listdir(test_dir):
                name_without_ext = os.path.splitext(f)[0]
                self.existing_files[name_without_ext] = f

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id'])
        img_id_clean = os.path.splitext(img_id)[0] 
        
        image = None
        
        if img_id_clean in self.existing_files:
            img_path = os.path.join(self.test_dir, self.existing_files[img_id_clean])
            image = cv2.imread(img_path) 
            
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        augmented_orig = self.transforms(image=image)
        img_orig = augmented_orig['image']
        
        image_flipped = cv2.flip(image, 1) 
        augmented_flip = self.transforms(image=image_flipped)
        img_flip = augmented_flip['image']

        return img_id, img_orig, img_flip

test_dataset = TestDatasetTTA(sub_df, TEST_DIR, transforms=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 6

# PENTING: GANTI dengan best_weights DARI TAHAP 4 TERBARU (OOF 10-Model)
best_weights = np.array([
    1.0, 1.0, 1.0, 1.0, 1.0, 1.0 
]) 

# ==========================================
# 3. PREDIKSI 10-MODEL ENSEMBLE DENGAN TTA
# ==========================================
print(f"Memulai prediksi {len(sub_df)} data dengan TTA (10-Model Ensemble Mode)...")

final_swin_probs = np.zeros((len(sub_df), NUM_CLASSES))
final_cnn_probs = np.zeros((len(sub_df), NUM_CLASSES))

# --- BAGIAN A: PREDIKSI 5 SWIN TRANSFORMER ---
print("\n=== EKSEKUSI 5 MODEL SWIN TRANSFORMER ===")
for fold in range(N_FOLDS):
    print(f"Memuat & Memprediksi dengan Swin Fold {fold}...")
    model_swin = create_swin_model()
    model_swin.load_state_dict(torch.load(f'best_swin_fold_{fold}.pth', map_location=device))
    model_swin.eval()

    fold_preds_swin = []
    with torch.no_grad():
        for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc=f"Swin TTA Fold {fold}"):
            imgs_orig = imgs_orig.to(device)
            imgs_flip = imgs_flip.to(device)

            out_orig = model_swin(imgs_orig)
            prob_orig = F.softmax(out_orig, dim=1)

            out_flip = model_swin(imgs_flip)
            prob_flip = F.softmax(out_flip, dim=1)

            prob_avg = (prob_orig + prob_flip) / 2.0
            fold_preds_swin.extend(prob_avg.cpu().numpy())

    final_swin_probs += (np.array(fold_preds_swin) / N_FOLDS)

    del model_swin
    torch.cuda.empty_cache()
    gc.collect()

# --- BAGIAN B: PREDIKSI 5 EFFICIENTNET V2 ---
print("\n=== EKSEKUSI 5 MODEL EFFICIENTNET V2 ===")
for fold in range(N_FOLDS):
    print(f"Memuat & Memprediksi dengan CNN Fold {fold}...")
    model_cnn = create_cnn_model()
    model_cnn.load_state_dict(torch.load(f'best_cnn_fold_{fold}.pth', map_location=device))
    model_cnn.eval()

    fold_preds_cnn = []
    with torch.no_grad():
        for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc=f"CNN TTA Fold {fold}"):
            imgs_orig = imgs_orig.to(device)
            imgs_flip = imgs_flip.to(device)

            out_orig = model_cnn(imgs_orig)
            prob_orig = F.softmax(out_orig, dim=1)

            out_flip = model_cnn(imgs_flip)
            prob_flip = F.softmax(out_flip, dim=1)

            prob_avg = (prob_orig + prob_flip) / 2.0
            fold_preds_cnn.extend(prob_avg.cpu().numpy())

    final_cnn_probs += (np.array(fold_preds_cnn) / N_FOLDS)

    del model_cnn
    torch.cuda.empty_cache()
    gc.collect()

# ==========================================
# 4. PENERAPAN KALIBRASI & PENYIMPANAN
# ==========================================
print("\n--- Menggabungkan 10 Prediksi, Menerapkan Kalibrasi & Menyusun Submission ---")

# Gabungkan dengan rasio kekuatan yang sama seperti di Tahap 4
final_ensemble_probs = (final_swin_probs * 0.60) + (final_cnn_probs * 0.40)

# Terapkan bobot kalibrasi (Powell) pada hasil rata-rata 10 model
weighted_probs = final_ensemble_probs * best_weights
final_preds = np.argmax(weighted_probs, axis=1)

# Simpan ke CSV
final_labels = [id2label[pred] for pred in final_preds]
sub_df['label'] = final_labels
sub_df.to_csv('submission.csv', index=False)

print("Selesai! File 'submission.csv' 10-Model Ensemble berhasil dibuat.")
print(sub_df.head())

Memulai prediksi 404 data dengan TTA (10-Model Ensemble Mode)...

=== EKSEKUSI 5 MODEL SWIN TRANSFORMER ===
Memuat & Memprediksi dengan Swin Fold 0...


Swin TTA Fold 0: 100%|██████████| 51/51 [00:33<00:00,  1.52it/s]


Memuat & Memprediksi dengan Swin Fold 1...


Swin TTA Fold 1: 100%|██████████| 51/51 [00:36<00:00,  1.41it/s]


Memuat & Memprediksi dengan Swin Fold 2...


...

Memuat & Memprediksi dengan CNN Fold 3...


CNN TTA Fold 3: 100%|██████████| 51/51 [00:14<00:00,  3.59it/s]


Memuat & Memprediksi dengan CNN Fold 4...


CNN TTA Fold 4: 100%|██████████| 51/51 [00:14<00:00,  3.55it/s]



--- Menggabungkan 10 Prediksi, Menerapkan Kalibrasi & Menyusun Submission ---
Selesai! File 'submission.csv' 10-Model Ensemble berhasil dibuat.
         id           label
0  test_001     fake_screen
1  test_002  fake_mannequin
2  test_003      realperson
3  test_004      realperson
4  test_005    fake_printed


In [ ]:
# ReLu